In [ ]:
import glob
import os

import pandas as pd

BASE = r"D:\self_study\Data Analysis\Findi360\Retail L2\DD-RET-002"  
OUT = "./Data"  
os.makedirs(OUT, exist_ok=True)                                                     

In [ ]:
def load(name):
    return pd.read_csv(glob.glob(f"{BASE}/*DD-RET-002_{name}.csv")[0])


def to_bool(col):
    return col.astype(str).str.strip().str.lower().eq("true")

In [ ]:
def clean_date(df):
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
    df["IsWeekend"] = to_bool(df["IsWeekend"])
    df["YearQuarter"] = df["Year"].astype(str) + "-" + df["Quarter"]
    return df


def clean_sales(df):
    for c in ["TransactionDateKey", "FulfillmentDateKey", "ReturnDateKey"]:
        df[c] = df[c].astype("Int64")          
    for c in ["ReturnFlag", "StockoutFlag"]:
        df[c] = to_bool(df[c])

    same_rows = [c for c in df.columns if c != "TransactionID"]  
    df["ReturnBeforeFulfilFlag"] = (df["ReturnDateKey"] < df["FulfillmentDateKey"]).fillna(False).astype(bool)
    df["ReturnDateCappedFlag"] = (df["ReturnDateKey"] == 20251231).fillna(False).astype(bool)
    df["PossibleDuplicateFlag"] = df.duplicated(subset=same_rows, keep=False)
    df["NegativeMarginFlag"] = df["GrossMarginSAR"] < 0
    return df


def clean_inventory(df):
    df["BelowReorderFlag"] = to_bool(df["BelowReorderFlag"])
    df["ZeroStockFlag"] = df["OnHandUnits"] == 0
    df["SnapshotDate"] = pd.to_datetime(df["SnapshotDateKey"].astype(str), format="%Y%m%d")
    df["SnapshotMonthEnd"] = df["SnapshotDate"] + pd.offsets.MonthEnd(0)
    return df


def clean_product(df, inventory):
    df["HasInventoryFlag"] = df["ProductID"].isin(inventory["ProductID"])
    return df

In [ ]:
inventory = clean_inventory(load("FactInventorySnapshot"))

tables = {
    "DimDate": clean_date(load("DimDate")),
    "DimStore": load("DimStore"),
    "DimProduct": clean_product(load("DimProduct"), inventory),
    "DimSupplier": load("DimSupplier"),
    "DimChannel": load("DimChannel"),
    "DimPromotion": load("DimPromotion"),
    "FactInventorySnapshot": inventory,
    "FactSalesTransactions": clean_sales(load("FactSalesTransactions")),
}

for name, df in tables.items():
    df.to_csv(f"{OUT}/{name}_clean.csv", index=False, date_format="%Y-%m-%d")